# LinCIR trên Google Colab Pro

Notebook này dùng **mã công bố chính thức** của LinCIR với CLIP ViT-L/14 để:

1. tải checkpoint `lincir_large.pt` và chạy smoke test thật;
2. đánh giá trên CIRR validation khi bạn đã đặt dataset trong Google Drive;
3. tùy chọn huấn luyện lại 20.000 bước và lưu checkpoint vào Drive.

Trước khi chạy, chọn **Runtime → Change runtime type → GPU**. Chạy lần lượt từ trên xuống. Mã nguồn trong repo LinCIR mang giấy phép **CC BY-NC 4.0**; hãy kiểm tra riêng model card trước khi dùng checkpoint ngoài nghiên cứu.

## 0. Gắn Drive và cấu hình

Để dùng Drive, giữ `USE_DRIVE = True`. Nếu Colab báo `credential propagation was unsuccessful`, cell sẽ tự dùng `/content/LinCIR` để bạn vẫn chạy được smoke test; dữ liệu trong `/content` sẽ mất khi runtime bị xóa. Sửa `CIRR_ROOT` nếu dataset của bạn nằm ở vị trí khác.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import hashlib
import json
import os
import shutil
import subprocess
import sys

import torch

USE_DRIVE = True
DRIVE_MOUNTED = False
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_MOUNTED = True
    except Exception as exc:
        print(f'Không gắn được Drive ({exc}). Tiếp tục bằng bộ nhớ tạm /content.')

if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Chọn Runtime → Change runtime type → GPU rồi chạy lại.')

WORK = Path('/content/lincir')
DRIVE_ROOT = Path('/content/drive/MyDrive/LinCIR') if DRIVE_MOUNTED else Path('/content/LinCIR')
CIRR_ROOT = (Path('/content/drive/MyDrive') if DRIVE_MOUNTED else Path('/content')) / 'datasets/CIRR'  # sửa nếu cần
CACHE_DIR = DRIVE_ROOT / 'hf_cache'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
RUN_DIR = DRIVE_ROOT / 'runs' / 'vit_l_official'
for directory in (CACHE_DIR, CHECKPOINT_DIR, RUN_DIR):
    directory.mkdir(parents=True, exist_ok=True)

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
TRAIN_BATCH_SIZE = 256 if vram_gb >= 35 else 128 if vram_gb >= 22 else 64
print(f'GPU: {gpu_name} ({vram_gb:.1f} GiB); batch huấn luyện đề xuất: {TRAIN_BATCH_SIZE}')


## 1. Lấy đúng phiên bản mã và cài phụ thuộc

Notebook khóa mã tác giả tại commit đã được kiểm tra. Hai vá nhỏ được áp dụng: sửa lỗi gõ `arg.max_grad_norm`, và chỉ dùng CC3M + Stable Diffusion Prompts như bài báo (bỏ Midjourney mà loader hiện tại đã thêm).

In [ ]:
REPO = 'https://github.com/navervision/lincir.git'
UPSTREAM_COMMIT = '1dec42d118da816be2a43fd43cc0746e05f63881'
if not WORK.exists():
    subprocess.run(['git', 'clone', REPO, str(WORK)], check=True)
subprocess.run(['git', '-C', str(WORK), 'checkout', UPSTREAM_COMMIT], check=True)

packages = [
    'transformers>=4.44,<5', 'datasets>=3,<5', 'accelerate>=0.30,<2',
    'spacy>=3.8,<4', 'huggingface_hub>=0.24,<2', 'webdataset>=0.2,<1',
    'ftfy>=6,<7', 'tensorboard>=2.16',
    'git+https://github.com/openai/CLIP.git',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)

train_file = WORK / 'train_phi.py'
train_text = train_file.read_text(encoding='utf-8')
train_text = train_text.replace('arg.max_grad_norm', 'args.max_grad_norm')
train_file.write_text(train_text, encoding='utf-8')

loader_file = WORK / 'loader.py'
loader_text = loader_file.read_text(encoding='utf-8')
loader_text = loader_text.replace(
    "                  'dataset3': 'Geonmo/midjourney-prompts-only',\n", ''
)
loader_file.write_text(loader_text, encoding='utf-8')

os.chdir(WORK)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
commit = subprocess.run(
    ['git', '-C', str(WORK), 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True
).stdout.strip()
assert commit == UPSTREAM_COMMIT
print('Mã nguồn sẵn sàng:', commit)


## 2. Tải model ViT-L do tác giả công bố

File được chép vào Drive, nên không phải tải lại sau khi Colab ngắt phiên.

In [ ]:
from huggingface_hub import hf_hub_download

MODEL_REVISION = '5118df4683de6efa09f8e5336d86ca626ed21c44'
EXPECTED_SHA256 = '5ba98d52db7a5e9a78f9ba7436991235477b89544caf246e4659642bf9b409bd'
downloaded = Path(hf_hub_download(
    repo_id='navervision/zeroshot-cir-models',
    filename='lincir_large.pt',
    revision=MODEL_REVISION,
    cache_dir=str(CACHE_DIR),
))
CHECKPOINT = CHECKPOINT_DIR / 'lincir_large.pt'
if not CHECKPOINT.exists():
    shutil.copy2(downloaded, CHECKPOINT)
if CHECKPOINT.stat().st_size < 1_000_000:
    raise RuntimeError('Checkpoint tải về có kích thước bất thường.')
with CHECKPOINT.open('rb') as stream:
    checkpoint_sha256 = hashlib.file_digest(stream, 'sha256').hexdigest()
if checkpoint_sha256 != EXPECTED_SHA256:
    raise RuntimeError(f'SHA256 checkpoint không khớp: {checkpoint_sha256}')
print(CHECKPOINT, f'{CHECKPOINT.stat().st_size / 2**20:.1f} MiB', checkpoint_sha256)


## 3. Smoke test checkpoint

Cell này chạy ảnh mẫu qua CLIP image encoder → Phi → CLIP text encoder. Nó kiểm tra model tải đúng và đầu ra hữu hạn; đây chưa phải đánh giá chất lượng.

In [ ]:
import clip
from PIL import Image
from encode_with_pseudo_tokens import encode_with_pseudo_tokens_HF
from models import Phi, build_text_encoder

device = torch.device('cuda')
args = SimpleNamespace(clip_model_name='large', mixed_precision='fp16', cache_dir=str(CACHE_DIR))
image_encoder, preprocess, text_encoder, _ = build_text_encoder(args)
phi = Phi(
    input_dim=text_encoder.config.projection_dim,
    hidden_dim=text_encoder.config.projection_dim * 4,
    output_dim=text_encoder.config.hidden_size,
    dropout=0.5,
)
state = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
phi.load_state_dict(state['Phi'])
image_encoder = image_encoder.to(device=device, dtype=torch.float16).eval()
text_encoder = text_encoder.to(device=device, dtype=torch.float16).eval()
phi = phi.to(device=device, dtype=torch.float16).eval()

with Image.open(WORK / 'example1.jpg') as image:
    pixels = preprocess(images=image.convert('RGB'), return_tensors='pt').pixel_values
pixels = pixels.to(device=device, dtype=torch.float16)
tokens = clip.tokenize(['a photo of $ that is a pencil sketch'], context_length=77).to(device)
with torch.inference_mode():
    image_features = image_encoder(pixel_values=pixels).image_embeds
    pseudo_token = phi(image_features)
    composed = encode_with_pseudo_tokens_HF(text_encoder, tokens, pseudo_token)
assert composed.shape == image_features.shape == (1, text_encoder.config.projection_dim)
assert torch.isfinite(composed).all()
print('PASS:', 'image', tuple(image_features.shape), 'pseudo-token', tuple(pseudo_token.shape),
      'composed', tuple(composed.shape))
del image_encoder, text_encoder, phi, state, pixels, tokens, image_features, pseudo_token, composed
torch.cuda.empty_cache()


## 4. Chuẩn bị CIRR

Tải CIRR theo hướng dẫn của dự án CIRR/SEARLE và đặt vào `CIRR_ROOT`. Cấu trúc tối thiểu cần có:

```text
CIRR/
├── cirr/
│   ├── captions/cap.rc2.val.json
│   └── image_splits/split.rc2.val.json
└── dev/... ảnh được các JSON tham chiếu ...
```

Dữ liệu không được đóng gói trong notebook vì điều khoản phân phối của dataset.

In [ ]:
caption_file = CIRR_ROOT / 'cirr/captions/cap.rc2.val.json'
split_file = CIRR_ROOT / 'cirr/image_splits/split.rc2.val.json'
CIRR_READY = caption_file.is_file() and split_file.is_file()
if CIRR_READY:
    records = json.loads(caption_file.read_text(encoding='utf-8'))
    image_paths = json.loads(split_file.read_text(encoding='utf-8'))
    missing = []
    for relative_path in image_paths.values():
        if not (CIRR_ROOT / relative_path).is_file():
            missing.append(relative_path)
            if len(missing) == 5:
                break
    if missing:
        raise FileNotFoundError(f'Thiếu ảnh CIRR, ví dụ: {missing}')
    print(f'CIRR hợp lệ: {len(records)} truy vấn, {len(image_paths)} ảnh gallery')
else:
    print('Chưa thấy CIRR tại', CIRR_ROOT)
    print('Hãy tải dataset, sửa CIRR_ROOT ở cell 0, rồi chạy lại cell này.')


## 5. Đánh giá checkpoint công bố trên CIRR validation

Kết quả được lưu thành log trong Drive. Đây là **validation/dev**, không nên so trực tiếp với số test trong Bảng 6 của bài báo.

In [ ]:
if not CIRR_READY:
    raise FileNotFoundError('CIRR chưa sẵn sàng; hoàn thành mục 4 trước.')
eval_command = [
    sys.executable, 'validate.py',
    '--eval-type', 'phi', '--dataset', 'cirr',
    '--dataset-path', str(CIRR_ROOT),
    '--phi-checkpoint-name', str(CHECKPOINT),
    '--clip_model_name', 'large', '--cache_dir', str(CACHE_DIR),
]
evaluation = subprocess.run(
    eval_command, cwd=WORK, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(evaluation.stdout)
(RUN_DIR / 'pretrained_cirr_val.log').write_text(evaluation.stdout, encoding='utf-8')
if evaluation.returncode:
    raise RuntimeError(f'Đánh giá thất bại, mã lỗi {evaluation.returncode}.')


## 6. Huấn luyện lại (tùy chọn)

Đổi `RUN_FULL_TRAINING=True` khi smoke test và CIRR validation đã chạy thành công. Cấu hình gần mã tác giả: ViT-L, fp16, 20.000 bước, kiểm tra/lưu mỗi 1.000 bước. Batch tự giảm theo VRAM nên kết quả có thể lệch bài báo.

Nếu `phi_latest.pt` đã tồn tại, notebook warm-start trọng số. Mã tác giả **không lưu optimizer/scheduler/dataloader state**, nên đây không phải resume bit-exact và bộ đếm bước chạy lại từ 0.

In [ ]:
RUN_FULL_TRAINING = False  # đổi thành True sau khi các mục trên đều PASS
MAX_TRAIN_STEPS = 20_000
if RUN_FULL_TRAINING:
    if not CIRR_READY:
        raise FileNotFoundError('CIRR chưa sẵn sàng; training của mã tác giả cần dev set để chọn model.')
    latest = RUN_DIR / 'checkpoints/phi_latest.pt'
    train_command = [
        sys.executable, 'train_phi.py',
        '--batch_size', str(TRAIN_BATCH_SIZE),
        '--output_dir', str(RUN_DIR),
        '--cirr_dataset_path', str(CIRR_ROOT),
        '--cache_dir', str(CACHE_DIR),
        '--mixed_precision', 'fp16',
        '--clip_model_name', 'large',
        '--validation_steps', '1000',
        '--checkpointing_steps', '1000',
        '--seed', '12345',
        '--num_workers', '2',
        '--lr_scheduler', 'constant_with_warmup',
        '--lr_warmup_steps', '0',
        '--gradient_accumulation_steps', '1',
        '--max_train_steps', str(MAX_TRAIN_STEPS),
    ]
    if latest.is_file():
        train_command += ['--resume', str(latest)]
        print('Warm-start từ', latest)
    subprocess.run(train_command, cwd=WORK, check=True)
else:
    print('Đã bỏ qua full training. Đổi RUN_FULL_TRAINING=True khi bạn sẵn sàng.')


## 7. Đánh giá model vừa huấn luyện

Cell chọn `phi_best.pt`, hoặc `phi_latest.pt` nếu chưa có best checkpoint.

In [ ]:
EVALUATE_TRAINED = False  # đổi thành True sau khi training tạo checkpoint
if EVALUATE_TRAINED:
    trained_checkpoint = RUN_DIR / 'checkpoints/phi_best.pt'
    if not trained_checkpoint.is_file():
        trained_checkpoint = RUN_DIR / 'checkpoints/phi_latest.pt'
    if not trained_checkpoint.is_file():
        raise FileNotFoundError('Chưa có checkpoint đã huấn luyện trong Drive.')
    trained_command = [
        sys.executable, 'validate.py',
        '--eval-type', 'phi', '--dataset', 'cirr',
        '--dataset-path', str(CIRR_ROOT),
        '--phi-checkpoint-name', str(trained_checkpoint),
        '--clip_model_name', 'large', '--cache_dir', str(CACHE_DIR),
    ]
    result = subprocess.run(
        trained_command, cwd=WORK, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print(result.stdout)
    (RUN_DIR / 'trained_cirr_val.log').write_text(result.stdout, encoding='utf-8')
    if result.returncode:
        raise RuntimeError(f'Đánh giá thất bại, mã lỗi {result.returncode}.')
else:
    print('Đã bỏ qua đánh giá model tự train.')


## Kết quả cần giữ lại

- Model công bố: `MyDrive/LinCIR/checkpoints/lincir_large.pt`
- Model tự train: `MyDrive/LinCIR/runs/vit_l_official/checkpoints/phi_best.pt`
- Log baseline: `MyDrive/LinCIR/runs/vit_l_official/pretrained_cirr_val.log`
- TensorBoard/checkpoint huấn luyện: bên trong thư mục run trên Drive

Khi baseline này chạy ổn định, hãy giữ nguyên seed/dataset/backbone/số bước và chỉ thay **một** thành phần cho mỗi thí nghiệm cải tiến.